# MPOSE2021 Scratch Encoder Pretraining

This notebook runs the full flow: install/check dependencies, download MPOSE2021 through the `mpose` package, convert it to the same 12-joint scratch encoder input, pretrain one or more scratch TCN/GCN encoder variants, and export `.keras`, `.tflite`, and metadata files.

The default run is intentionally small. Increase `EPOCHS`, `STEPS_PER_EPOCH`, and remove the sample caps for a real pretrain run. Use `PRETRAIN_CONFIGS` to train multiple variants like scratch training.


In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'scripts' / 'pretrain_mpose2021.py').exists():
            return path
    raise RuntimeError('Could not find pjt_main project root')

PROJECT_ROOT = find_project_root()
print('PROJECT_ROOT =', PROJECT_ROOT)
print('PYTHON =', sys.executable)

PROJECT_ROOT = /workspace/users/yijin/boot_env/pjt_main
PYTHON = /workspace/users/yijin/boot_env/.venv/bin/python


In [2]:
# Install the optional downloader if needed. TensorFlow is expected in the training env.
if importlib.util.find_spec('mpose') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'mpose==1.2'])
else:
    print('mpose is already installed')

import tensorflow as tf
print('tensorflow', tf.__version__)

mpose is already installed


2026-04-16 09:57:11.989341: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-16 09:57:12.004867: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-16 09:57:12.004890: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-16 09:57:12.015068: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-16 09:57:12.579972: W tensorflow/compiler/tf

tensorflow 2.16.2


In [3]:
# Dataset and multi-model pretrain settings
POSE_EXTRACTOR = 'posenet'  # posenet uses 17 keypoints and is smaller than openpose
SPLIT = 1

# Smoke defaults. Set both caps to None for a real run.
MAX_TRAIN_SAMPLES = 3000
MAX_VAL_SAMPLES = 800
EPOCHS = 20
STEPS_PER_EPOCH = 80
VALIDATION_STEPS = 20
BATCH_SIZE = 64
PATIENCE = 5
LEARNING_RATE = 1e-3

# Set ONLY to a list of names to train a subset, for example ['mpose2021_gcn_e64_triplet'].
ONLY = []

PRETRAIN_CONFIGS = [
    {
        'name': 'mpose2021_tcn_e32_triplet',
        'model_type': 'tcn',
        'embedding_dim': 32,
        'loss_type': 'triplet',
    },
    {
        'name': 'mpose2021_tcn_e64_triplet',
        'model_type': 'tcn',
        'embedding_dim': 64,
        'loss_type': 'triplet',
    },
    {
        'name': 'mpose2021_gcn_e32_triplet',
        'model_type': 'gcn',
        'embedding_dim': 32,
        'loss_type': 'triplet',
    },
    {
        'name': 'mpose2021_gcn_e64_triplet',
        'model_type': 'gcn',
        'embedding_dim': 64,
        'loss_type': 'triplet',
    },
    # Optional BCE baselines. Uncomment if you want pair-classification variants too.
    # {'name': 'mpose2021_gcn_e64_bce', 'model_type': 'gcn', 'embedding_dim': 64, 'loss_type': 'bce'},
    # {'name': 'mpose2021_tcn_e64_bce', 'model_type': 'tcn', 'embedding_dim': 64, 'loss_type': 'bce'},
]

SELECTED_CONFIGS = [cfg for cfg in PRETRAIN_CONFIGS if not ONLY or cfg['name'] in ONLY]
assert SELECTED_CONFIGS, 'No configs selected'

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'models' / 'pretrain' / 'mpose2021'
CACHE_PATH = PROJECT_ROOT / 'data' / 'pretrain' / 'mpose2021' / f'mpose2021_{POSE_EXTRACTOR}_split{SPLIT}_scratch.npz'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

print('CACHE_PATH =', CACHE_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('selected configs:')
for cfg in SELECTED_CONFIGS:
    print(' -', cfg)


CACHE_PATH = /workspace/users/yijin/boot_env/pjt_main/data/pretrain/mpose2021/mpose2021_posenet_split1_scratch.npz
OUTPUT_DIR = /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/mpose2021
selected configs:
 - {'name': 'mpose2021_tcn_e32_triplet', 'model_type': 'tcn', 'embedding_dim': 32, 'loss_type': 'triplet'}
 - {'name': 'mpose2021_tcn_e64_triplet', 'model_type': 'tcn', 'embedding_dim': 64, 'loss_type': 'triplet'}
 - {'name': 'mpose2021_gcn_e32_triplet', 'model_type': 'gcn', 'embedding_dim': 32, 'loss_type': 'triplet'}
 - {'name': 'mpose2021_gcn_e64_triplet', 'model_type': 'gcn', 'embedding_dim': 64, 'loss_type': 'triplet'}


In [4]:
# Download + convert only. Re-run with OVERWRITE_CACHE=True to refresh the cache.
# This cache is shared by all variants below.
OVERWRITE_CACHE = False
cmd = [
    sys.executable, str(PROJECT_ROOT / 'scripts' / 'pretrain_mpose2021.py'),
    '--pose-extractor', POSE_EXTRACTOR,
    '--split', str(SPLIT),
    '--converted-output', str(CACHE_PATH),
    '--prepare-only',
]
if MAX_TRAIN_SAMPLES is not None:
    cmd += ['--max-train-samples', str(MAX_TRAIN_SAMPLES)]
if MAX_VAL_SAMPLES is not None:
    cmd += ['--max-val-samples', str(MAX_VAL_SAMPLES)]
if OVERWRITE_CACHE:
    cmd += ['--overwrite-cache']
print(' '.join(cmd))
subprocess.check_call(cmd, cwd=PROJECT_ROOT)


/workspace/users/yijin/boot_env/.venv/bin/python /workspace/users/yijin/boot_env/pjt_main/scripts/pretrain_mpose2021.py --pose-extractor posenet --split 1 --converted-output /workspace/users/yijin/boot_env/pjt_main/data/pretrain/mpose2021/mpose2021_posenet_split1_scratch.npz --prepare-only --max-train-samples 3000 --max-val-samples 800
[LOAD] converted MPOSE cache: /workspace/users/yijin/boot_env/pjt_main/data/pretrain/mpose2021/mpose2021_posenet_split1_scratch.npz
[DATA] train: (3000, 30, 12, 2) labels= {0:100, 1:48, 2:74, 3:60, 4:70, 5:66, 6:48, 7:447...} classes=20
[DATA] val:   (800, 30, 12, 2) labels= {0:50, 1:11, 2:14, 3:12, 4:23, 5:18, 6:27, 7:85...} classes=20
[DONE] prepared MPOSE cache


0

In [5]:
# Inspect converted data.
import numpy as np
data = np.load(CACHE_PATH)
print('train_windows', data['train_windows'].shape, data['train_windows'].dtype)
print('train_labels ', data['train_labels'].shape, sorted(set(data['train_labels'].tolist()))[:10])
print('val_windows  ', data['val_windows'].shape, data['val_windows'].dtype)
print('val_labels   ', data['val_labels'].shape, sorted(set(data['val_labels'].tolist()))[:10])

train_windows (3000, 30, 12, 2) float32
train_labels  (3000,) [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
val_windows   (800, 30, 12, 2) float32
val_labels    (800,) [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [6]:
# Train + export every selected variant. This uses --train-only so the cached data is reused.
# Outputs:
#   data/models/pretrain/mpose2021/{name}_encoder.keras
#   data/models/pretrain/mpose2021/{name}.tflite
#   data/models/pretrain/mpose2021/{name}_meta.json

import json
import time

TRAIN_RESULTS = []

for index, cfg in enumerate(SELECTED_CONFIGS, start=1):
    print('=' * 100)
    print(f"[{index}/{len(SELECTED_CONFIGS)}] Pretraining {cfg['name']}")
    print('=' * 100)

    cmd = [
        sys.executable, str(PROJECT_ROOT / 'scripts' / 'pretrain_mpose2021.py'),
        '--train-only',
        '--pose-extractor', POSE_EXTRACTOR,
        '--split', str(SPLIT),
        '--converted-output', str(CACHE_PATH),
        '--model-name', cfg['name'],
        '--model-type', cfg['model_type'],
        '--output-dir', str(OUTPUT_DIR),
        '--loss-type', cfg.get('loss_type', 'triplet'),
        '--embedding-dim', str(cfg.get('embedding_dim', 64)),
        '--epochs', str(cfg.get('epochs', EPOCHS)),
        '--steps-per-epoch', str(cfg.get('steps_per_epoch', STEPS_PER_EPOCH)),
        '--validation-steps', str(cfg.get('validation_steps', VALIDATION_STEPS)),
        '--batch-size', str(cfg.get('batch_size', BATCH_SIZE)),
        '--patience', str(cfg.get('patience', PATIENCE)),
        '--learning-rate', str(cfg.get('learning_rate', LEARNING_RATE)),
    ]
    if 'filters' in cfg:
        cmd += ['--filters', str(cfg['filters'])]
    if 'blocks' in cfg:
        cmd += ['--blocks', str(cfg['blocks'])]
    if 'dropout' in cfg:
        cmd += ['--dropout', str(cfg['dropout'])]
    if 'triplet_margin' in cfg:
        cmd += ['--triplet-margin', str(cfg['triplet_margin'])]
    if cfg.get('no_quantize', False):
        cmd += ['--no-quantize']

    started = time.time()
    print(' '.join(cmd))
    subprocess.check_call(cmd, cwd=PROJECT_ROOT)
    elapsed = time.time() - started

    meta_path = OUTPUT_DIR / f"{cfg['name']}_meta.json"
    meta = json.loads(meta_path.read_text(encoding='utf-8')) if meta_path.exists() else {}
    TRAIN_RESULTS.append({
        'name': cfg['name'],
        'model_type': cfg['model_type'],
        'embedding_dim': cfg.get('embedding_dim', 64),
        'loss_type': cfg.get('loss_type', 'triplet'),
        'elapsed_sec': round(elapsed, 1),
        'training_summary': meta.get('training_summary', {}),
        'smoke_metrics': meta.get('smoke_metrics', {}),
        'keras': str(OUTPUT_DIR / f"{cfg['name']}_encoder.keras"),
        'tflite': str(OUTPUT_DIR / f"{cfg['name']}.tflite"),
        'meta': str(meta_path),
    })

print()
print('All selected pretraining runs finished.')
for result in TRAIN_RESULTS:
    print(result)


[1/4] Pretraining mpose2021_tcn_e32_triplet
/workspace/users/yijin/boot_env/.venv/bin/python /workspace/users/yijin/boot_env/pjt_main/scripts/pretrain_mpose2021.py --train-only --pose-extractor posenet --split 1 --converted-output /workspace/users/yijin/boot_env/pjt_main/data/pretrain/mpose2021/mpose2021_posenet_split1_scratch.npz --model-name mpose2021_tcn_e32_triplet --model-type tcn --output-dir /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/mpose2021 --loss-type triplet --embedding-dim 32 --epochs 20 --steps-per-epoch 80 --validation-steps 20 --batch-size 64 --patience 5 --learning-rate 0.001
[LOAD] converted MPOSE cache: /workspace/users/yijin/boot_env/pjt_main/data/pretrain/mpose2021/mpose2021_posenet_split1_scratch.npz
[DATA] train: (3000, 30, 12, 2) labels= {0:100, 1:48, 2:74, 3:60, 4:70, 5:66, 6:48, 7:447...} classes=20
[DATA] val:   (800, 30, 12, 2) labels= {0:50, 1:11, 2:14, 3:12, 4:23, 5:18, 6:27, 7:85...} classes=20


2026-04-16 09:59:04.454920: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-16 09:59:04.470358: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-16 09:59:04.470380: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-16 09:59:05.082526: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2026-04-16 09:59:06.392532: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the gu

Model: "scratch_triplet_similarity"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ anchor_window       │ (None, 30, 12, 2) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼��───────────┼───────────────────┤
│ positive_window     │ (None, 30, 12, 2) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ negative_window     │ (None, 30, 12, 2) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼────────────────���──┤
│ scratch_tcn_encoder │ (None, 32)   

2026-04-16 09:59:43.852693: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
W0000 00:00:1776301184.063292 1715523 tf_tfl_flatbuffer_helpers.cc:390] Ignored output_format.
W0000 00:00:1776301184.063318 1715523 tf_tfl_flatbuffer_helpers.cc:393] Ignored drop_control_dependency.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


[SAVE] TFLite encoder: /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/mpose2021/mpose2021_tcn_e32_triplet.tflite (97.1 KiB)
[VERIFY] input=[1, 30, 12, 2] output=[1, 32] embedding_norm=7.1145
[BEST] {'monitor': 'val_loss', 'restore_best_weights': True, 'epochs_ran': 20, 'best_epoch': 17, 'best_val_loss': 0.07905253022909164, 'best_epoch_loss': 0.05536575987935066}
[METRIC] {'same_cosine_mean': 0.8364259298541583, 'different_cosine_mean': 0.5545604311555508, 'margin_mean': 0.28186549869860755}
[SAVE] metadata: /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/mpose2021/mpose2021_tcn_e32_triplet_meta.json

Fine-tune starting point:
  /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/mpose2021/mpose2021_tcn_e32_triplet_encoder.keras
[2/4] Pretraining mpose2021_tcn_e64_triplet
/workspace/users/yijin/boot_env/.venv/bin/python /workspace/users/yijin/boot_env/pjt_main/scripts/pretrain_mpose2021.py --train-only --pose-extractor posenet --split 1 --converte

...

[SAVE] TFLite encoder: /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/mpose2021/mpose2021_gcn_e32_triplet.tflite (466.4 KiB)
[VERIFY] input=[1, 30, 12, 2] output=[1, 32] embedding_norm=27.1287
[BEST] {'monitor': 'val_loss', 'restore_best_weights': True, 'epochs_ran': 10, 'best_epoch': 5, 'best_val_loss': 0.07667890936136246, 'best_epoch_loss': 0.08609666675329208}
[METRIC] {'same_cosine_mean': 0.8873718458926305, 'different_cosine_mean': 0.6213183189975098, 'margin_mean': 0.26605352689512074}
[SAVE] metadata: /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/mpose2021/mpose2021_gcn_e32_triplet_meta.json

Fine-tune starting point:
  /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/mpose2021/mpose2021_gcn_e32_triplet_encoder.keras
[4/4] Pretraining mpose2021_gcn_e64_triplet
/workspace/users/yijin/boot_env/.venv/bin/python /workspace/users/yijin/boot_env/pjt_main/scripts/pretrain_mpose2021.py --train-only --pose-extractor posenet --split 1 --convert

2026-04-16 10:03:35.606813: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-16 10:03:35.621812: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-16 10:03:35.621837: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-16 10:03:36.246449: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2026-04-16 10:03:37.615146: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the gu

Model: "scratch_triplet_similarity"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ anchor_window       │ (None, 30, 12, 2) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼��───────────┼───────────────────┤
│ positive_window     │ (None, 30, 12, 2) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ negative_window     │ (None, 30, 12, 2) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼────────────────���──┤
│ scratch_gcn_encoder │ (None, 64)   

2026-04-16 10:08:50.521402: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
W0000 00:00:1776301730.759971 1772241 tf_tfl_flatbuffer_helpers.cc:390] Ignored output_format.
W0000 00:00:1776301730.759997 1772241 tf_tfl_flatbuffer_helpers.cc:393] Ignored drop_control_dependency.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


[SAVE] TFLite encoder: /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/mpose2021/mpose2021_gcn_e64_triplet.tflite (468.5 KiB)
[VERIFY] input=[1, 30, 12, 2] output=[1, 64] embedding_norm=164.7517
[BEST] {'monitor': 'val_loss', 'restore_best_weights': True, 'epochs_ran': 20, 'best_epoch': 20, 'best_val_loss': 0.0639452412724495, 'best_epoch_loss': 0.048424676060676575}
[METRIC] {'same_cosine_mean': 0.8101892629056238, 'different_cosine_mean': 0.398078741003701, 'margin_mean': 0.41211052190192277}
[SAVE] metadata: /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/mpose2021/mpose2021_gcn_e64_triplet_meta.json

Fine-tune starting point:
  /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/mpose2021/mpose2021_gcn_e64_triplet_encoder.keras

All selected pretraining runs finished.
{'name': 'mpose2021_tcn_e32_triplet', 'model_type': 'tcn', 'embedding_dim': 32, 'loss_type': 'triplet', 'elapsed_sec': 50.3, 'training_summary': {'monitor': 'val_loss', 'restore_b

In [7]:
# Check generated artifacts and compact result table.
from pathlib import Path

if 'TRAIN_RESULTS' not in globals():
    TRAIN_RESULTS = []
    for cfg in SELECTED_CONFIGS:
        meta_path = OUTPUT_DIR / f"{cfg['name']}_meta.json"
        meta = json.loads(meta_path.read_text(encoding='utf-8')) if meta_path.exists() else {}
        TRAIN_RESULTS.append({
            'name': cfg['name'],
            'model_type': cfg['model_type'],
            'embedding_dim': cfg.get('embedding_dim', 64),
            'loss_type': cfg.get('loss_type', 'triplet'),
            'training_summary': meta.get('training_summary', {}),
            'smoke_metrics': meta.get('smoke_metrics', {}),
            'keras': str(OUTPUT_DIR / f"{cfg['name']}_encoder.keras"),
            'tflite': str(OUTPUT_DIR / f"{cfg['name']}.tflite"),
            'meta': str(meta_path),
        })

for result in TRAIN_RESULTS:
    print('-' * 100)
    print(result['name'])
    print('  best:', result.get('training_summary', {}))
    print('  smoke:', result.get('smoke_metrics', {}))
    for key in ('keras', 'tflite', 'meta'):
        path = Path(result[key])
        size = f'{path.stat().st_size / 1024:.1f} KiB' if path.exists() else 'missing'
        print(f'  {key}: {path} ({size})')


----------------------------------------------------------------------------------------------------
mpose2021_tcn_e32_triplet
  best: {'monitor': 'val_loss', 'restore_best_weights': True, 'epochs_ran': 20, 'best_epoch': 17, 'best_val_loss': 0.07905253022909164, 'best_epoch_loss': 0.05536575987935066}
  smoke: {'same_cosine_mean': 0.8364259298541583, 'different_cosine_mean': 0.5545604311555508, 'margin_mean': 0.28186549869860755}
  keras: /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/mpose2021/mpose2021_tcn_e32_triplet_encoder.keras (429.6 KiB)
  tflite: /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/mpose2021/mpose2021_tcn_e32_triplet.tflite (97.1 KiB)
  meta: /workspace/users/yijin/boot_env/pjt_main/data/models/pretrain/mpose2021/mpose2021_tcn_e32_triplet_meta.json (5.3 KiB)
----------------------------------------------------------------------------------------------------
mpose2021_tcn_e64_triplet
  best: {'monitor': 'val_loss', 'restore_best_weights'

Each `.keras` encoder is a starting point for dance fine-tuning. The usual next step is to compare the variants on dance-specific validation: from-scratch scratch models, MPOSE triplet variants, and optional MPOSE BCE variants. Pick the best encoder as the initialization for AIST++ or local dance fine-tuning, then export the final `.tflite` model for service runtime.
